# Experiment 11: Layer 0 Joint Tucker Adaptation Evaluation

This notebook applies the **optimal Tucker rank configurations** discovered during the Experiment 10 1%-sweep to **Layer 0** of `google/gemma-3-1b-it` **simultaneously** (joint weight injection) and evaluates end-to-end performance:

### Discovered Optimal Configurations (Layer 0 Attention Projections):
| Submodule | Original Weight | Chunking Config | Discovered Optimal Ranks | Individual Param Cut | Individual Recon Err |
|:---|:---:|:---:|:---:|:---:|:---:|
| `self_attn.q_proj` | `[1024, 1152]` | $K=4, M=240$ | **`[4, 200, 468]`** | $+13.05\%$ | $36.15\%$ |
| `self_attn.k_proj` | `[256, 1152]` | $K=4, M=60$ | **`[4, 43, 136]`** | $+33.93\%$ | $52.07\%$ |
| `self_attn.v_proj` | `[256, 1152]` | $K=4, M=60$ | **`[4, 46, 153]`** | $+25.06\%$ | $51.33\%$ |
| `self_attn.o_proj` | `[1152, 1024]` | $K=4, M=250$ | **`[4, 208, 441]`** | $+14.99\%$ | $40.66\%$ |

*(Note: `self_attn`, `q_norm`, `k_norm`, and `mlp` were activation profiling targets and 1D scales; the 4 linear projection weights above constitute the full linear weight manifold of Layer 0 Attention).*

### Objectives:
1. Load `google/gemma-3-1b-it` in verified FP32.
2. Evaluate **Pristine Baseline** on MNLI (500 samples) and generate baseline Chocolate Cake recipe.
3. Apply nested DBSCAN clustering + Tucker decomposition with the exact discovered ranks to all 4 projections **simultaneously** into Layer 0.
4. Output the **Full Chocolate Cake Recipe** (untruncated).
5. Evaluate **MNLI on 500 samples** to observe joint downstream accuracy impact.


In [ ]:
# =====================================================================
# STEP 1: Environment Setup & Imports
# =====================================================================
import os
import sys
import time
from typing import Dict, List, Any

import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.cluster import DBSCAN
from sklearn.metrics import accuracy_score
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor
from transformers import AutoModelForCausalLM, AutoTokenizer

# Writable cache setup
os.environ["TRITON_CACHE_DIR"] = os.path.expanduser("~/.triton_cache")
os.makedirs(os.environ["TRITON_CACHE_DIR"], exist_ok=True)
os.environ["HF_DATASETS_OFFLINE"] = "1"

tl.set_backend("pytorch")
torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")



In [ ]:
# =====================================================================
# STEP 2: Load Model in Verified FP32
# =====================================================================
MODEL_ID = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32, device_map=device)
model.eval()

l0 = model.model.layers[0]

# Verify FP32 precision
actual_dtype = next(model.parameters()).dtype
total_params = sum(x.numel() for x in model.parameters())
assert actual_dtype == torch.float32, f"Expected float32 but got {actual_dtype}"
print(f"Loaded {MODEL_ID}")
print(f"  Dtype  : {actual_dtype}")
print(f"  Params : {total_params:,} ({total_params/1e9:.3f}B)")
print(f"  VRAM   : {total_params * 4 / 1024**3:.2f} GiB (weights-only estimate)")



In [ ]:
# =====================================================================
# STEP 3: Evaluation Helpers (MNLI 500 Samples & Chocolate Cake Prompt)
# =====================================================================
EVAL_SAMPLES = 500

print(f"Loading GLUE MNLI validation_matched ({EVAL_SAMPLES} samples)...")
ds = load_dataset("nyu-mll/glue", "mnli", split="validation_matched").select(range(EVAL_SAMPLES))
labels_names = ["entailment", "neutral", "contradiction"]
label_token_ids = [tokenizer.encode(" " + n, add_special_tokens=False)[0] for n in labels_names]

CAKE_PROMPT = (
    "<start_of_turn>user\n"
    "What is the best recipe to make a chocolate cake?<end_of_turn>\n"
    "<start_of_turn>model\n"
)

def evaluate_mnli(model_to_eval, desc="Evaluating MNLI") -> float:
    model_to_eval.eval()
    preds, gt = [], []
    with torch.no_grad():
        for sample in ds:
            prompt = (
                f"<start_of_turn>user\n"
                f"Premise: {sample['premise']}\n"
                f"Hypothesis: {sample['hypothesis']}\n"
                f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
                f"Answer with one word\n"
                f"<start_of_turn>model\n"
            )
            inp = tokenizer(prompt, return_tensors="pt").to(model_to_eval.device)
            out = model_to_eval(**inp, logits_to_keep=1)
            preds.append(torch.argmax(out.logits[0, -1, :][label_token_ids]).item())
            gt.append(sample["label"])
    return float(accuracy_score(gt, preds))

def generate_cake_recipe_full(model_to_eval, max_new_tokens=1024) -> str:
    model_to_eval.eval()
    inp = tokenizer(CAKE_PROMPT, return_tensors="pt").to(model_to_eval.device)
    with torch.no_grad():
        tokens = model_to_eval.generate(
            **inp,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )
    return tokenizer.decode(tokens[0][inp.input_ids.shape[1]:], skip_special_tokens=True)



In [ ]:
# =====================================================================
# STEP 4: Pristine FP32 Baseline Evaluation
# =====================================================================
print("Running Pristine FP32 Baseline MNLI (500 samples)...")
baseline_acc = evaluate_mnli(model)
print(f"\n>>> PRISTINE BASELINE MNLI ACCURACY (N=500): {baseline_acc * 100:.2f}%\n")

print("Generating Pristine Baseline Chocolate Cake Recipe...")
baseline_cake_recipe = generate_cake_recipe_full(model, max_new_tokens=1024)
print("=" * 80)
print("PRISTINE BASELINE CHOCOLATE CAKE RECIPE:")
print("=" * 80)
print(baseline_cake_recipe)
print("=" * 80)



In [ ]:
# =====================================================================
# STEP 5: Collect Calibration Activations for Guided DBSCAN Clustering
# =====================================================================
# Run a calibration pass across 100 samples to collect feature activation statistics
CALIB_SAMPLES = 100
calib_store = {
    "q_proj": [],
    "k_proj": [],
    "v_proj": [],
    "o_proj": [],
}

def make_hook(key):
    def hook(m, inp, out):
        t = out[0] if isinstance(out, tuple) else out
        flat = t.detach().cpu().float().reshape(-1, t.shape[-1])
        calib_store[key].append(flat)
    return hook

calib_hooks = [
    l0.self_attn.q_proj.register_forward_hook(make_hook("q_proj")),
    l0.self_attn.k_proj.register_forward_hook(make_hook("k_proj")),
    l0.self_attn.v_proj.register_forward_hook(make_hook("v_proj")),
    l0.self_attn.o_proj.register_forward_hook(make_hook("o_proj")),
]

print(f"Collecting calibration activations on {CALIB_SAMPLES} samples...")
with torch.no_grad():
    for sample in ds.select(range(CALIB_SAMPLES)):
        prompt = (
            f"<start_of_turn>user\n"
            f"Premise: {sample['premise']}\n"
            f"Hypothesis: {sample['hypothesis']}\n"
            f"Determine if the relationship between the 'Premise' and 'Hypothesis' is 'entailment', 'neutral' or 'contradiction.'\n"
            f"Answer with one word\n"
            f"<start_of_turn>model\n"
        )
        inp = tokenizer(prompt, return_tensors="pt").to(model.device)
        _ = model(**inp, logits_to_keep=1)

for h in calib_hooks:
    h.remove()

calib_acts = {}
for k, v in calib_store.items():
    cat = torch.cat(v, dim=0).numpy()
    calib_acts[k] = cat[:5000]
    print(f"  {k:<8} activations shape: {calib_acts[k].shape}")



In [ ]:
# =====================================================================
# STEP 6: Nested DBSCAN Clustering for Weight Matrices
# =====================================================================
def nested_clustering_weight(
    act_matrix: np.ndarray,
    weight_tensor: torch.Tensor,
    chunk_size: int,
    num_chunks: int,
    n_iter: int = 3,
    z_cutoff: float = 3.0,
) -> Dict[str, Any]:
    out_dim, in_dim = weight_tensor.shape

    if act_matrix is not None and act_matrix.shape[1] == out_dim:
        v = np.mean(act_matrix, axis=0)
        variances = np.var(act_matrix, axis=0)
    else:
        w_np = weight_tensor.detach().cpu().float().numpy()
        v = np.mean(w_np, axis=1)
        variances = np.var(w_np, axis=1)

    z = np.abs((v - np.mean(v)) / (np.std(v) + 1e-8))
    var99 = float(np.quantile(variances, 0.99)) if out_dim > 10 else 1e9
    super_mask = (z > z_cutoff) | (variances >= var99)
    super_coords = np.where(super_mask)[0]

    candidate_idx = np.where(~super_mask)[0]
    eff_chunk = chunk_size
    if len(candidate_idx) < num_chunks * eff_chunk:
        eff_chunk = max(5, len(candidate_idx) // num_chunks)

    chunk_list = []

    for _ in range(n_iter):
        if len(candidate_idx) < eff_chunk or len(chunk_list) >= num_chunks:
            break
        v_sub = v[candidate_idx]
        eps = max(0.02, float(np.std(v_sub) * 0.18))
        min_s = max(5, min(20, eff_chunk // 4))
        db = DBSCAN(eps=eps, min_samples=min_s, metric="euclidean")
        labels = db.fit_predict(v_sub.reshape(-1, 1))
        for lab in [l for l in np.unique(labels) if l != -1]:
            c_local = np.where(labels == lab)[0]
            if len(c_local) >= eff_chunk:
                srt = c_local[np.argsort(v_sub[c_local])]
                for ci in range(len(srt) // eff_chunk):
                    chunk_list.append(candidate_idx[srt[ci * eff_chunk:(ci + 1) * eff_chunk]])
                    if len(chunk_list) >= num_chunks:
                        break
            if len(chunk_list) >= num_chunks:
                break
        assigned = set(np.concatenate(chunk_list)) if chunk_list else set()
        candidate_idx = np.array([i for i in candidate_idx if i not in assigned])

    if len(chunk_list) < num_chunks:
        assigned = set(np.concatenate(chunk_list)) if chunk_list else set()
        avail = [i for i in range(out_dim) if i not in assigned and i not in super_coords]
        for _ in range(num_chunks - len(chunk_list)):
            if len(avail) >= eff_chunk:
                chunk_list.append(np.array(avail[:eff_chunk]))
                avail = avail[eff_chunk:]
            else:
                break

    if not chunk_list:
        raise RuntimeError(f"Weight clustering failed: out_dim={out_dim}, eff_chunk={eff_chunk}")

    T = torch.stack([weight_tensor[c, :].float().cpu() for c in chunk_list], dim=0)

    return {
        "tensor": T,
        "chunk_list": chunk_list,
        "super_coords": super_coords,
        "eff_chunk": eff_chunk,
        "num_chunks": len(chunk_list),
        "out_dim": out_dim,
        "in_dim": in_dim,
    }



In [ ]:
# =====================================================================
# STEP 7: Apply All Discovered Tucker Ranks Simultaneously to Layer 0
# =====================================================================
# Discovered configurations from Experiment 10:
JOINT_TUCKER_CONFIGS = {
    "q_proj": {
        "module": l0.self_attn.q_proj,
        "chunk_size": 240,
        "num_chunks": 4,
        "target_ranks": [4, 200, 468],
        "desc": "q_proj [1024, 1152]",
    },
    "k_proj": {
        "module": l0.self_attn.k_proj,
        "chunk_size": 60,
        "num_chunks": 4,
        "target_ranks": [4, 43, 136],
        "desc": "k_proj [256, 1152]",
    },
    "v_proj": {
        "module": l0.self_attn.v_proj,
        "chunk_size": 60,
        "num_chunks": 4,
        "target_ranks": [4, 46, 153],
        "desc": "v_proj [256, 1152]",
    },
    "o_proj": {
        "module": l0.self_attn.o_proj,
        "chunk_size": 250,
        "num_chunks": 4,
        "target_ranks": [4, 208, 441],
        "desc": "o_proj [1152, 1024]",
    },
}

adaptation_stats = {}

print("=" * 80)
print("APPLYING JOINT TUCKER ADAPTATION TO LAYER 0 ATTENTION")
print("=" * 80)

for name, cfg in JOINT_TUCKER_CONFIGS.items():
    mod = cfg["module"]
    W_orig = mod.weight.data.clone()
    
    # 1. Cluster weight into 3D tensor
    cdata = nested_clustering_weight(
        act_matrix=calib_acts[name],
        weight_tensor=W_orig,
        chunk_size=cfg["chunk_size"],
        num_chunks=cfg["num_chunks"],
    )
    T = cdata["tensor"]
    ranks = cfg["target_ranks"]
    
    # 2. Decompose with Tucker using exact target ranks
    core, factors = tucker(T, rank=ranks, init="svd")
    T_hat = tucker_to_tensor((core, factors))
    
    # 3. Compute reconstruction error and parameter compression
    rel_err = (torch.norm(T - T_hat) / torch.norm(T)).item()
    orig_p = T.numel()
    comp_p = core.numel() + sum(f.numel() for f in factors)
    cut_pct = (orig_p - comp_p) / orig_p * 100.0
    
    # 4. Inject reconstructed weights (quarantining superweights)
    # NOTE: Injected weights are PERMANENTLY KEPT (no restore)!
    for k_idx, c in enumerate(cdata["chunk_list"]):
        mod.weight.data[c, :] = T_hat[k_idx].to(mod.weight.device, dtype=mod.weight.dtype)
        
    n_super = len(cdata["super_coords"])
    out_d = cdata["out_dim"]
    
    adaptation_stats[name] = {
        "desc": cfg["desc"],
        "tensor_shape": list(T.shape),
        "target_ranks": ranks,
        "recon_err_pct": round(rel_err * 100.0, 2),
        "params_cut_pct": round(cut_pct, 2),
        "quarantined_superweights": f"{n_super}/{out_d} ({n_super/out_d*100:.1f}%)",
    }
    
    print(f"✓ {name:<8} ({cfg['desc']}):")
    print(f"    3D Tensor Shape: {list(T.shape)} -> Tucker Ranks: {ranks}")
    print(f"    Recon Error    : {rel_err * 100.0:.2f}%")
    print(f"    Param Cut      : {cut_pct:+.2f}%")
    print(f"    Superweights   : {n_super}/{out_d} ({n_super/out_d*100:.1f}%) preserved in FP32")

print("\n>>> ALL 4 PROJECTIONS SUCCESSFULLY ADAPTED SIMULTANEOUSLY ON LAYER 0!")



In [ ]:
# =====================================================================
# STEP 8: Full Chocolate Cake Recipe Generation (Joint Tucker Adapted)
# =====================================================================
print("Generating full chocolate cake recipe with Joint Tucker Adapted Layer 0...")
print("=" * 80)
print("JOINT TUCKER ADAPTED CHOCOLATE CAKE RECIPE:")
print("=" * 80)

adapted_cake_recipe = generate_cake_recipe_full(model, max_new_tokens=1024)
print(adapted_cake_recipe)
print("=" * 80)



In [ ]:
# =====================================================================
# STEP 9: Evaluate MNLI on 500 Samples (Joint Tucker Adapted)
# =====================================================================
print("Evaluating MNLI (500 samples) on Joint Tucker Adapted model...")
joint_acc = evaluate_mnli(model)

print("\n" + "=" * 80)
print("LAYER 0 JOINT TUCKER ADAPTATION — FINAL RESULTS")
print("=" * 80)
print(f"Pristine Baseline Accuracy (N=500)     : {baseline_acc * 100:.2f}%")
print(f"Joint Tucker Adapted Accuracy (N=500)  : {joint_acc * 100:.2f}%")
print(f"Delta                                  : {(joint_acc - baseline_acc) * 100:+.2f}%")
print("=" * 80)

print("\nSubmodule Breakdown:")
print(f"{'Submodule':<10} | {'Tensor Shape':<18} | {'Tucker Ranks':<16} | {'Recon Err%':>10} | {'Param Cut%':>11} | {'Superweights':<16}")
print("-" * 90)
for k, v in adaptation_stats.items():
    print(f"{k:<10} | {str(v['tensor_shape']):<18} | {str(v['target_ranks']):<16} | {v['recon_err_pct']:>10.2f} | {v['params_cut_pct']:>+11.2f} | {v['quarantined_superweights']:<16}")
print("=" * 90)



In [ ]:
# =====================================================================
# STEP 10: Save Evaluation Results to JSON
# =====================================================================
results_payload = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model_id": MODEL_ID,
    "eval_samples": EVAL_SAMPLES,
    "baseline_mnli_accuracy_pct": round(baseline_acc * 100.0, 2),
    "joint_adapted_mnli_accuracy_pct": round(joint_acc * 100.0, 2),
    "accuracy_delta_pct": round((joint_acc - baseline_acc) * 100.0, 2),
    "adaptation_settings": adaptation_stats,
    "baseline_cake_recipe": baseline_cake_recipe,
    "adapted_cake_recipe": adapted_cake_recipe,
}

out_dir = "experiments/01_layer_based_bench/artifacts"
os.makedirs(out_dir, exist_ok=True)
out_file = f"{out_dir}/11_layer_0_joint_evaluation_results.json"
with open(out_file, "w") as f:
    json.dump(results_payload, f, indent=2)

print(f"Saved evaluation artifacts to: {out_file}")

